In [1]:
import os
import time
import timm
import random
import numpy as np
from PIL import Image, ImageEnhance

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

from tqdm import tqdm
from torchvision import transforms, models

from datetime import datetime
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ==========================================
# SET SEED TOÀN DIỆN
# ==========================================
def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [3]:
# ==========================================
# AUGMENTATION
# ==========================================
def augment(image: Image.Image,
            mask: Image.Image,
            flip_prob: float = 0.5,
            rotate_prob: float = 0.5,
            rotate_range: tuple = (-15, 15),
            brightness_prob: float = 0.5,
            brightness_range: tuple = (0.8, 1.2),
            contrast_prob: float = 0.5,
            contrast_range: tuple = (0.8, 1.2),
            gamma_prob: float = 0.5,
            gamma_range: tuple = (0.8, 1.2),
            noise_prob: float = 0.3,
            noise_std: float = 0.02) -> tuple:

    # 1. Geometric transforms (đồng bộ image & mask)
    # Lật ngang
    if random.random() < flip_prob:
        image = image.transpose(Image.FLIP_LEFT_RIGHT)
        mask = mask.transpose(Image.FLIP_LEFT_RIGHT)

    # Lật dọc
    if random.random() < flip_prob:
        image = image.transpose(Image.FLIP_TOP_BOTTOM)
        mask = mask.transpose(Image.FLIP_TOP_BOTTOM)

    # Xoay
    if random.random() < rotate_prob:
        angle = random.uniform(*rotate_range)
        image = image.rotate(angle, resample=Image.BILINEAR, fillcolor=0)
        mask = mask.rotate(angle, resample=Image.NEAREST, fillcolor=0)

    # 2. Intensity transforms (Chỉ image)
    # Độ sáng
    if random.random() < brightness_prob:
        factor = random.uniform(*brightness_range)
        image = ImageEnhance.Brightness(image).enhance(factor)

    # Độ tương phản
    if random.random() < contrast_prob:
        factor = random.uniform(*contrast_range)
        image = ImageEnhance.Contrast(image).enhance(factor)

    # Độ phơi sáng
    if random.random() < gamma_prob:
        gamma = random.uniform(*gamma_range)
        image = image.point(lambda p: 255 * ((p / 255.0) ** gamma))

    # 3. Gaussian Noise (Chỉ image) - mô phỏng nhiễu X-quang thật
    if random.random() < noise_prob:
        img_np = np.array(image).astype(np.float32)
        noise = np.random.normal(0, noise_std * 255, img_np.shape)
        img_np = np.clip(img_np + noise, 0, 255).astype(np.uint8)
        image = Image.fromarray(img_np)

    return image, mask

In [4]:
# ==========================================
# SEGMENTATION DATASET
# ==========================================
def convert_to_rgb(img: Image.Image) -> Image.Image:
    return img.convert('RGB')

def convert_to_grayscale_3ch(img: Image.Image) -> Image.Image:
    return img.convert('L').convert('RGB')

class SegmentationDataset(Dataset):
    def __init__(self,
                 image_dir: str,
                 mask_dir: str,
                 mean: list,
                 std: list,
                 is_rgb: bool = False,
                 img_size: tuple = (448, 448),
                 augment_flag: bool = True,
                 augment_params: dict = None):

        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.is_rgb = is_rgb
        self.img_size = img_size
        self.augment_flag = augment_flag

        # Tham số augmentation mặc định
        self.augment_params = {
            'flip_prob': 0.5,
            'rotate_prob': 0.5,
            'rotate_range': (-15, 15),
            'brightness_prob': 0.5,
            'brightness_range': (0.8, 1.2),
            'contrast_prob': 0.5,
            'contrast_range': (0.8, 1.2),
            'gamma_prob': 0.5,
            'gamma_range': (0.8, 1.2),
            'noise_prob': 0.3,
            'noise_std': 0.02,
        }
        if augment_params:
            self.augment_params.update(augment_params)

        self.image_names = sorted(os.listdir(image_dir))
        self.mask_names = sorted(os.listdir(mask_dir))

        assert len(self.image_names) == len(self.mask_names), \
            f"Số ảnh ({len(self.image_names)}) != số mask ({len(self.mask_names)})"

        # Transform Image
        if is_rgb:
            pil_convert = transforms.Lambda(convert_to_rgb)
        else:
            pil_convert = transforms.Lambda(convert_to_grayscale_3ch)

        self.image_transform = transforms.Compose([
            pil_convert,
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ])

        # Transform Mask
        self.mask_transform = transforms.Compose([
            transforms.Resize(img_size, interpolation=transforms.InterpolationMode.NEAREST),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_names[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_names[idx])

        image = Image.open(img_path).convert('RGB' if self.is_rgb else 'L')
        mask = Image.open(mask_path).convert('L')

        if self.augment_flag:
            image, mask = augment(image, mask, **self.augment_params)

        image = self.image_transform(image)
        mask = self.mask_transform(mask)
        mask = (mask > 0.5).float()

        return image, mask

In [5]:
# ==========================================
# GET DATALOADERS
# ==========================================
def get_dataloaders(dataset_stats, img_size=(448, 448), batch_size=16, num_workers=2):
    base_path = dataset_stats['path']
    train_dataset = SegmentationDataset(
        image_dir=os.path.join(base_path, 'train', 'images'),
        mask_dir=os.path.join(base_path, 'train', 'masks'),
        mean=dataset_stats['mean'], std=dataset_stats['std'],
        is_rgb=dataset_stats['is_rgb'], img_size=img_size,
        augment_flag=True
    )
    val_dataset = SegmentationDataset(
        image_dir=os.path.join(base_path, 'valid', 'images'),
        mask_dir=os.path.join(base_path, 'valid', 'masks'),
        mean=dataset_stats['mean'], std=dataset_stats['std'],
        is_rgb=dataset_stats['is_rgb'], img_size=img_size,
        augment_flag=False
    )

    generator = torch.Generator().manual_seed(42)   # seed cố định cho generator

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True,
                              worker_init_fn=seed_worker, generator=generator)

    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=True,
                            worker_init_fn=seed_worker, generator=generator)

    print(f"Dataset: {os.path.basename(base_path.rstrip(os.sep))}")
    print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Image size: {img_size}\n")
    return train_loader, val_loader

In [6]:
# ==========================================
# BASELINE - UNET - GITHUB - TRAIN FROM SCRATCH
# ==========================================
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    """Two consecutive conv + BN + ReLU"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        else:
            self.up = nn.ConvTranspose2d(in_channels // 2, in_channels // 2, kernel_size=2, stride=2)

        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        # Padding để match kích thước (nếu cần)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])

        # Concatenate
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, n_classes):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, n_classes, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, bilinear=True):
        super(UNet, self).__init__()

        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 512)

        self.up1 = Up(1024, 256, bilinear)
        self.up2 = Up(512, 128, bilinear)
        self.up3 = Up(256, 64, bilinear)
        self.up4 = Up(128, 64, bilinear)

        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)

        logits = self.outc(x)

        return logits  # Dùng cho binary segmentation

In [7]:
# ==========================================
# CE-NET MODEL
# ==========================================
class DACblock(nn.Module):
    def __init__(self, channel):
        super().__init__()
        self.dilate1 = nn.Conv2d(channel, channel, 3, dilation=1, padding=1)
        self.dilate2 = nn.Conv2d(channel, channel, 3, dilation=3, padding=3)
        self.dilate3 = nn.Conv2d(channel, channel, 3, dilation=5, padding=5)
        self.conv1x1 = nn.Conv2d(channel, channel, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        d1 = self.relu(self.dilate1(x))
        d2 = self.relu(self.conv1x1(self.dilate2(x)))
        d3 = self.relu(self.conv1x1(self.dilate2(self.dilate1(x))))
        d4 = self.relu(self.conv1x1(self.dilate3(self.dilate2(self.dilate1(x)))))
        return x + d1 + d2 + d3 + d4


class RMPblock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.pool1 = nn.MaxPool2d(2, stride=2)
        self.pool2 = nn.MaxPool2d(3, stride=3)
        self.pool3 = nn.MaxPool2d(5, stride=5)
        self.pool4 = nn.MaxPool2d(6, stride=6)
        self.conv = nn.Conv2d(in_channels, 1, 1)

    def forward(self, x):
        size = x.shape[2:]
        p1 = F.interpolate(self.conv(self.pool1(x)), size=size, mode='bilinear', align_corners=True)
        p2 = F.interpolate(self.conv(self.pool2(x)), size=size, mode='bilinear', align_corners=True)
        p3 = F.interpolate(self.conv(self.pool3(x)), size=size, mode='bilinear', align_corners=True)
        p4 = F.interpolate(self.conv(self.pool4(x)), size=size, mode='bilinear', align_corners=True)
        return torch.cat([x, p1, p2, p3, p4], dim=1)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels // 4, 1)
        self.bn1 = nn.BatchNorm2d(in_channels // 4)
        self.deconv = nn.ConvTranspose2d(in_channels // 4, in_channels // 4, 3, stride=2, padding=1, output_padding=1)
        self.bn2 = nn.BatchNorm2d(in_channels // 4)
        self.conv2 = nn.Conv2d(in_channels // 4, out_channels, 1)
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.deconv(x)))
        x = self.relu(self.bn3(self.conv2(x)))
        return x


class CENet(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        resnet = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)

        self.firstconv = resnet.conv1
        self.firstbn = resnet.bn1
        self.firstrelu = resnet.relu
        self.firstmaxpool = resnet.maxpool

        self.encoder1 = resnet.layer1
        self.encoder2 = resnet.layer2
        self.encoder3 = resnet.layer3
        self.encoder4 = resnet.layer4

        self.dac = DACblock(512)
        self.rmp = RMPblock(512)

        self.decoder4 = DecoderBlock(516, 256)
        self.decoder3 = DecoderBlock(512, 128)
        self.decoder2 = DecoderBlock(256, 64)
        self.decoder1 = DecoderBlock(128, 64)

        self.finaldeconv1 = nn.ConvTranspose2d(128, 32, 4, stride=2, padding=1)
        self.finalconv2 = nn.Conv2d(32, 32, 3, padding=1)
        self.finalconv3 = nn.Conv2d(32, num_classes, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        e0 = self.firstrelu(self.firstbn(self.firstconv(x)))
        x = self.firstmaxpool(e0)

        e1 = self.encoder1(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)

        dac_out = self.dac(e4)
        rmp_out = self.rmp(dac_out)

        d4 = self.decoder4(rmp_out)
        d4_cat = torch.cat([d4, e3], dim=1)

        d3 = self.decoder3(d4_cat)
        d3_cat = torch.cat([d3, e2], dim=1)

        d2 = self.decoder2(d3_cat)
        d2_cat = torch.cat([d2, e1], dim=1)

        d1 = self.decoder1(d2_cat)
        d1_cat = torch.cat([d1, e0], dim=1)

        out = self.relu(self.finaldeconv1(d1_cat))
        out = self.relu(self.finalconv2(out))
        out = self.finalconv3(out)

        return out

# ==========================================
# BIẾN THỂ
# ==========================================
class CENet_EffB4(nn.Module):
    """
    CENet with EfficientNet-B4 encoder (MBConv / depthwise-separable).
    Encoder channels: e0=24  e1=32  e2=56  e3=160  e4=448
    Best accuracy-to-speed ratio; ~19 M total params.
    """
    def __init__(self, num_classes=1, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained=pretrained,
            features_only=True,
            out_indices=(0, 1, 2, 3, 4),
        )
        # 448×448 verified:
        #   stage0: (B,  24, 224, 224)
        #   stage1: (B,  32, 112, 112)
        #   stage2: (B,  56,  56,  56)
        #   stage3: (B, 160,  28,  28)
        #   stage4: (B, 448,  14,  14)

        self.dac = DACblock(448)
        self.rmp = RMPblock(448)    # → 452ch

        self.decoder4 = DecoderBlock(452, 160)
        self.decoder3 = DecoderBlock(320, 56)   # 160 + 160 skip
        self.decoder2 = DecoderBlock(112, 32)   # 56  + 56  skip
        self.decoder1 = DecoderBlock(64,  24)   # 32  + 32  skip

        # Final upsample ×2 + ×2 back to full resolution
        self.finaldeconv1 = nn.ConvTranspose2d(48, 32, 4, stride=2, padding=1)
        self.finalconv2   = nn.Conv2d(32, 32, 3, padding=1)
        self.finalconv3   = nn.Conv2d(32, num_classes, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        e0, e1, e2, e3, e4 = self.backbone(x)

        rmp_out = self.rmp(self.dac(e4))                       # 452ch

        d4 = self.decoder4(rmp_out)                            # 160ch
        d3 = self.decoder3(torch.cat([d4, e3], 1))            # 56ch
        d2 = self.decoder2(torch.cat([d3, e2], 1))            # 32ch
        d1 = self.decoder1(torch.cat([d2, e1], 1))            # 24ch

        out = self.relu(self.finaldeconv1(torch.cat([d1, e0], 1)))  # 48→32, ×2
        out = self.relu(self.finalconv2(out))
        out = self.finalconv3(out)
        return out


class CENet_MobileV3(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'mobilenetv3_large_100',
            pretrained=pretrained,
            features_only=True,
            out_indices=(0, 1, 2, 3, 4),
        )
        # 448×448 verified:
        #   stage0: (B,  16, 224, 224)
        #   stage1: (B,  24, 112, 112)
        #   stage2: (B,  40,  56,  56)
        #   stage3: (B, 112,  28,  28)
        #   stage4: (B, 960,  14,  14)

        # Project 960 → 448 trước DAC (tránh DAC quá nặng)
        self.bottleneck_conv = nn.Conv2d(960, 448, 1)
        self.bottleneck_bn   = nn.BatchNorm2d(448)
        self.bottleneck_relu = nn.ReLU(inplace=True)

        self.dac = DACblock(448)
        self.rmp = RMPblock(448)              # → 452ch

        self.decoder4 = DecoderBlock(452, 112)
        self.decoder3 = DecoderBlock(224,  40)  # 112 + e3(112) = 224
        self.decoder2 = DecoderBlock( 80,  24)  #  40 + e2( 40) =  80
        self.decoder1 = DecoderBlock( 48,  16)  #  24 + e1( 24) =  48

        # cat(d1=16, e0=16) = 32ch → ×2 → full resolution
        self.finaldeconv1 = nn.ConvTranspose2d(32, 32, 4, stride=2, padding=1)
        self.finalconv2   = nn.Conv2d(32, 32, 3, padding=1)
        self.finalconv3   = nn.Conv2d(32, num_classes, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # Encoder
        e0, e1, e2, e3, e4 = self.backbone(x)
        #  16ch/2   24ch/4  40ch/8  112ch/16  960ch/32

        # Bottleneck projection: 960 → 448
        e4 = self.bottleneck_relu(self.bottleneck_bn(self.bottleneck_conv(e4)))

        # DAC + RMP
        dac_out = self.dac(e4)
        rmp_out = self.rmp(dac_out)  # 260ch

        # Decoder
        d4 = self.decoder4(rmp_out)                       # 112ch, /16
        d3 = self.decoder3(torch.cat([d4, e3], dim=1))   #  40ch, /8
        d2 = self.decoder2(torch.cat([d3, e2], dim=1))   #  24ch, /4
        d1 = self.decoder1(torch.cat([d2, e1], dim=1))   #  16ch, /2

        # Final head
        out = self.relu(self.finaldeconv1(torch.cat([d1, e0], dim=1)))  # 32ch, /1
        out = self.relu(self.finalconv2(out))
        out = self.finalconv3(out)
        return out


class CENet_SwinT(nn.Module):
    """
    CENet với Swin Transformer-Tiny encoder.

    Swin-T stage channels : e0=96  e1=192  e2=384  e3=768
    Spatial resolution    : e0=H/4  e1=H/8  e2=H/16  e3=H/32

    DecoderBlock tự upsample ×2 bên trong (ConvTranspose2d stride=2)
    nên sau 4 decoder: H/32 → H/16 → H/8 → H/4 → H/2
    Chỉ cần 1× finaldeconv để về H (giống CENet gốc).
    """
    def __init__(self, num_classes=1, img_size=512, pretrained=True):
        super().__init__()

        # ── Encoder ────────────────────────────────────────────────
        self.backbone = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=pretrained,
            features_only=True,
            img_size=img_size,
            out_indices=(0, 1, 2, 3),
        )
        ec = self.backbone.feature_info.channels()  # [96, 192, 384, 768]

        # ── Context Extractor ──────────────────────────────────────
        self.dac = DACblock(ec[3])        # 768 → 768
        self.rmp = RMPblock(ec[3])        # 768 → 772

        # ── Decoder ────────────────────────────────────────────────
        # Mỗi DecoderBlock tự upsample ×2 bên trong
        # decoder4: 772       → 384,  H/32 → H/16;  cat e2(384) → 768
        # decoder3: 768       → 192,  H/16 → H/8;   cat e1(192) → 384
        # decoder2: 384       → 96,   H/8  → H/4;   cat e0(96)  → 192
        # decoder1: 192       → 96,   H/4  → H/2    (không skip)
        self.decoder4 = DecoderBlock(ec[3] + 4, ec[2])   # 772 → 384
        self.decoder3 = DecoderBlock(ec[2] * 2, ec[1])   # 768 → 192
        self.decoder2 = DecoderBlock(ec[1] * 2, ec[0])   # 384 → 96
        self.decoder1 = DecoderBlock(ec[0] * 2, ec[0])   # 192 → 96

        # ── Final head ─────────────────────────────────────────────
        # d1 ra H/2 → 1× ConvTranspose2d → H (giống CENet gốc)
        self.finaldeconv1 = nn.ConvTranspose2d(ec[0], 32, 4, stride=2, padding=1)
        self.finalconv2   = nn.Conv2d(32, 32, 3, padding=1)
        self.finalconv3   = nn.Conv2d(32, num_classes, 1)
        self.relu         = nn.ReLU(inplace=True)

    @staticmethod
    def _bchw(feats):
        """Swin trả về (B,H,W,C) → (B,C,H,W)."""
        return [f.permute(0, 3, 1, 2).contiguous() for f in feats]

    def forward(self, x):
        # ── Encoder ────────────────────────────────────────────────
        e0, e1, e2, e3 = self._bchw(self.backbone(x))
        # e0: B×96 ×H/4 ×W/4
        # e1: B×192×H/8 ×W/8
        # e2: B×384×H/16×W/16
        # e3: B×768×H/32×W/32

        # ── Context Extractor ──────────────────────────────────────
        dac_out = self.dac(e3)
        rmp_out = self.rmp(dac_out)   # B×772×H/32×W/32

        # ── Decoder + skip ─────────────────────────────────────────
        d4     = self.decoder4(rmp_out)            # B×384×H/16
        d4_cat = torch.cat([d4, e2], dim=1)        # B×768×H/16

        d3     = self.decoder3(d4_cat)             # B×192×H/8
        d3_cat = torch.cat([d3, e1], dim=1)        # B×384×H/8

        d2     = self.decoder2(d3_cat)             # B×96×H/4
        d2_cat = torch.cat([d2, e0], dim=1)        # B×192×H/4

        d1     = self.decoder1(d2_cat)             # B×96×H/2

        # ── Final head ─────────────────────────────────────────────
        out = self.relu(self.finaldeconv1(d1))     # B×32×H
        out = self.relu(self.finalconv2(out))      # B×32×H
        return self.finalconv3(out)                # B×num_classes×H×W

In [11]:
# ==========================================
# LOSS FUNCTIONS
# ==========================================
class DiceLoss(nn.Module):
    """Dice Loss thuần - Gần với paper CE-Net"""
    def __init__(self, smooth=1e-5):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(-1)
        targets = targets.view(-1)

        intersection = (probs * targets).sum()
        dice = (2. * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1 - dice

In [12]:
# ==========================================
# METRICS THEO PAPER CE-Net
# ==========================================
def compute_metrics(preds, targets, smooth=1e-5):
    """
    preds: raw logits (N, 1, H, W)
    targets: binary mask (N, 1, H, W)
    Trả về: Overlapping Error (E), Sensitivity (Sen), Accuracy (Acc), Dice
    """
    preds = (torch.sigmoid(preds) > 0.5).float()

    preds = preds.view(-1)
    targets = targets.view(-1)

    TP = (preds * targets).sum()
    FP = (preds * (1 - targets)).sum()
    FN = ((1 - preds) * targets).sum()
    TN = ((1 - preds) * (1 - targets)).sum()

    # Overlapping Error E = 1 - |S ∩ G| / |S ∪ G|
    intersection = TP
    union = TP + FP + FN
    overlapping_error = 1 - (intersection + smooth) / (union + smooth) if union > 0 else 1.0

    # Sensitivity (Sen)
    sensitivity = TP / (TP + FN + smooth)

    # Accuracy (Acc)
    accuracy = (TP + TN) / (TP + TN + FP + FN + smooth)

    # Dice (để tham khảo)
    dice = (2. * TP + smooth) / (2. * TP + FP + FN + smooth)

    return {
        'E': overlapping_error.item(),
        'Sen': sensitivity.item(),
        'Acc': accuracy.item(),
        'Dice': dice.item()
    }

In [13]:
DATASET_CONFIGS = {
    "Shenzhen": {
        'path': '/content/drive/MyDrive/DS200_BIG_DATA/Datasets/Shenzhen',
        'mean': [0.6143, 0.6143, 0.6143],
        'std': [0.2564, 0.2564, 0.2564],
        'is_rgb': True,
        'note': 'RGB - 512x512'
    },
    "JSRT": {
        'path': '/content/drive/MyDrive/DS200_BIG_DATA/Datasets/JSRT',
        'mean': [0.595, 0.595, 0.595],
        'std': [0.2743, 0.2743, 0.2743],
        'is_rgb': True,
        'note': 'RGB - 224x224'
    },
    "LUNA": {
        'path': '/content/drive/MyDrive/DS200_BIG_DATA/Datasets/LUNA',
        'mean': [0.8118, 0.8118, 0.8118],
        'std': [0.3791, 0.3791, 0.3791],
        'is_rgb': False,   # Grayscale
        'note': 'Grayscale - 512x512'
    }
}

CENET_REGISTRY = {
    'r34':       CENet,
    'effb4':     CENet_EffB4, #12
    'mobilev3': CENet_MobileV3, #24
    'swint':    CENet_SwinT,
}

In [20]:
# ==========================================
# HÀM CHẠY MỘT EXPERIMENT
# ==========================================
dataset_name = "LUNA"
size = 448
num_runs = 5
loss_func_name = "dice"
model_name = "swin"

# Tạo folder result
result_dir = f"/content/drive/MyDrive/DS200_BIG_DATA/result_{dataset_name}"
os.makedirs(result_dir, exist_ok=True)
result_file = os.path.join(result_dir, f"result_{model_name}_{loss_func_name}.txt")

def get_dataset_stats(name):
    """Tự động lấy stats theo dataset name"""
    if name not in DATASET_CONFIGS:
        raise ValueError(f"Dataset '{name}' chưa được định nghĩa. Các dataset hỗ trợ: {list(DATASET_CONFIGS.keys())}")
    return DATASET_CONFIGS[name]

def run_one_experiment(seed, run_id, num_epochs=20):
    print(f"\n{'='*80}")
    print(f"RUN {run_id + 1}/{num_runs} - Seed: {seed}")
    print(f"{'='*80}\n")

    set_all_seeds(seed)

    dataset_stats = get_dataset_stats(dataset_name)

    train_loader, val_loader = get_dataloaders(dataset_stats, img_size=(448, 448), batch_size=14, num_workers=2)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if model_name == "effb4":
        print("Using EfficientNet-B4 encoder")
        model = CENet_EffB4(num_classes=1).to(device)
    elif model_name == "mobilev3":
        print("Using MobileNetV3 encoder")
        model = CENet_MobileV3(num_classes=1).to(device)
    elif model_name == "unet":
        print("Using Unet - baseline")
        model = UNet(n_channels=3, n_classes=1, bilinear=True).to(device)
    elif model_name == "swin":
        print("Using Swin Transformer encoder")
        model = CENet_SwinT(num_classes=1, img_size=448).to(device)
    else:
        print("Using ResNet34 encoder")
        model = CENet(num_classes=1).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    criterion = DiceLoss()

    best_val_e = 1.0
    best_epoch = 0
    best_metrics = None
    run_start_time = time.time()
    epoch_times = []  # lưu thời gian từng epoch

    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        model.train()
        epoch_loss = 0.0
        for images, masks in tqdm(train_loader, desc=f"Run {run_id+1} Epoch {epoch+1}", leave=False):
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_train_loss = epoch_loss / len(train_loader)

        # Validation
        model.eval()
        val_loss = 0.0
        val_metrics = {'E': 0.0, 'Sen': 0.0, 'Acc': 0.0, 'Dice': 0.0}
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                loss = criterion(outputs, masks)
                val_loss += loss.item()
                batch_metrics = compute_metrics(outputs, masks)
                for k in val_metrics:
                    val_metrics[k] += batch_metrics[k]

        num_batches = len(val_loader)
        avg_val_loss = val_loss / num_batches
        avg_metrics = {k: v / num_batches for k, v in val_metrics.items()}

        epoch_time = time.time() - epoch_start_time
        epoch_times.append(epoch_time)

        # Save best model
        if avg_metrics['E'] < best_val_e:
            best_val_e = avg_metrics['E']
            best_epoch = epoch + 1
            best_metrics = avg_metrics.copy()
            # torch.save(model.state_dict(), os.path.join(save_dir, f"best_model_E_run{run_id+1}.pth"))
            print(f"  >>> BEST at epoch {best_epoch} | E = {best_val_e:.4f}")

        print(f"Epoch {epoch+1:2d} | Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "                     # ← thêm Val Loss
              f"Val E: {avg_metrics['E']:.4f} | Sen: {avg_metrics['Sen']:.4f} | "
              f"Acc: {avg_metrics['Acc']:.4f} | Dice: {avg_metrics['Dice']:.4f}"
              f"Time: {epoch_time:.1f}s")

    total_time = time.time() - run_start_time
    avg_epoch_time = np.mean(epoch_times)

    print(f"\nRun {run_id+1} completed in {total_time/60:.1f} minutes "
          f"({total_time:.1f}s) | Avg epoch: {avg_epoch_time:.1f}s")
    print(f"Best E = {best_metrics['E']:.4f} at epoch {best_epoch}\n")

    best_metrics['total_time'] = total_time
    best_metrics['avg_epoch_time'] = avg_epoch_time
    best_metrics['best_epoch'] = best_epoch

    if run_id == 0:
        save_path = os.path.join(
            result_dir,
            f"best_model_{model_name}_{loss_func_name}.pth"
        )

        torch.save({
            'epoch': best_epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': best_metrics,
            'seed': seed
        }, save_path)

        print(f"Model saved to: {save_path}")

    return best_metrics, best_epoch

In [21]:
# ==========================================
# CHẠY num_run RUNS
# ==========================================
seeds = list(range(42, 42 + num_runs))   # [42, 43, ..., 46]
all_best_metrics = []
run_summaries = []

for run_id in range(num_runs):
    metrics, best_epoch = run_one_experiment(seeds[run_id], run_id, num_epochs=20)
    all_best_metrics.append(metrics)

    # Lưu thông tin run để ghi vào file
    run_summaries.append(
        f"Run {run_id+1} (Seed: {seeds[run_id]}) - Best Epoch: {best_epoch}\n"
        f"   E: {metrics['E']:.4f} | Sen: {metrics['Sen']:.4f} | "
        f"Acc: {metrics['Acc']:.4f} | Dice: {metrics['Dice']:.4f}\n"
    )


RUN 1/5 - Seed: 42

Dataset: LUNA
Train: 210 | Val: 53 | Image size: (448, 448)

Using Swin Transformer encoder


Epoch  1 | Train Loss: 0.6793 | Val Loss: 0.6893 | Val E: 1.0000 | Sen: 0.0000 | Acc: 0.7711 | Dice: 0.0000Time: 27.7s


  >>> BEST at epoch 2 | E = 0.4210
Epoch  2 | Train Loss: 0.6411 | Val Loss: 0.6598 | Val E: 0.4210 | Sen: 0.9096 | Acc: 0.8492 | Dice: 0.7330Time: 27.9s


  >>> BEST at epoch 3 | E = 0.0629
Epoch  3 | Train Loss: 0.5293 | Val Loss: 0.4980 | Val E: 0.0629 | Sen: 0.9759 | Acc: 0.9851 | Dice: 0.9675Time: 28.1s


  >>> BEST at epoch 4 | E = 0.0462
Epoch  4 | Train Loss: 0.3150 | Val Loss: 0.1951 | Val E: 0.0462 | Sen: 0.9800 | Acc: 0.9892 | Dice: 0.9764Time: 27.9s


...

Epoch 16 | Train Loss: 0.0197 | Val Loss: 0.0198 | Val E: 0.0343 | Sen: 0.9894 | Acc: 0.9920 | Dice: 0.9825Time: 28.2s


  >>> BEST at epoch 17 | E = 0.0333
Epoch 17 | Train Loss: 0.0190 | Val Loss: 0.0191 | Val E: 0.0333 | Sen: 0.9888 | Acc: 0.9923 | Dice: 0.9831Time: 28.9s


Epoch 18 | Train Loss: 0.0189 | Val Loss: 0.0190 | Val E: 0.0333 | Sen: 0.9832 | Acc: 0.9923 | Dice: 0.9830Time: 28.9s


  >>> BEST at epoch 19 | E = 0.0330
Epoch 19 | Train Loss: 0.0183 | Val Loss: 0.0187 | Val E: 0.0330 | Sen: 0.9834 | Acc: 0.9924 | Dice: 0.9832Time: 29.3s


  >>> BEST at epoch 20 | E = 0.0328
Epoch 20 | Train Loss: 0.0179 | Val Loss: 0.0184 | Val E: 0.0328 | Sen: 0.9831 | Acc: 0.9924 | Dice: 0.9833Time: 29.2s

Run 5 completed in 9.5 minutes (570.5s) | Avg epoch: 28.5s
Best E = 0.0328 at epoch 20



In [22]:
# ==========================================
# TÍNH MEAN ± STD & LƯU TỔNG KẾT CUỐI
# ==========================================
print("\n" + "="*90)
print(f"FINAL STATISTICAL RESULTS ({num_runs} independent runs)")
print("="*90)

with open(result_file, 'w', encoding='utf-8') as f:
    f.write(f"CE-Net - {dataset_name} - {num_runs} Independent Runs\n")
    f.write(f"Loss Function: {loss_func_name}\n")
    f.write(f"Image Size: 448x448\n")
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("="*90 + "\n\n")

    f.write("BEST RESULTS PER RUN:\n")
    f.write("-" * 70 + "\n")

    for i, metrics in enumerate(all_best_metrics):
        f.write(f"Run {i+1} (Seed: {seeds[i]}) - Best Epoch: {metrics['best_epoch']}\n")
        f.write(f" E: {metrics['E']:.4f} | Sen: {metrics['Sen']:.4f} | "
                f"Acc: {metrics['Acc']:.4f} | Dice: {metrics['Dice']:.4f}\n")
        f.write(f" Total Time : {metrics['total_time']/60:.1f} phút "
                f"({metrics['total_time']:.1f}s) | "
                f"Avg Epoch: {metrics['avg_epoch_time']:.1f}s\n\n")

    f.write("\n" + "="*90 + "\n")
    f.write("FINAL MEAN ± STD:\n")
    f.write("="*90 + "\n")

    metrics_names = ['E', 'Sen', 'Acc', 'Dice']
    for metric in metrics_names:
        values = [m[metric] for m in all_best_metrics]
        mean_val = np.mean(values)
        std_val = np.std(values, ddof=1)
        line = f"{metric:>12}: {mean_val:.4f} ± {std_val:.4f}"
        print(line)
        f.write(line + "\n")

    # Thêm thời gian trung bình
    avg_total_time = np.mean([m['total_time'] for m in all_best_metrics])
    avg_epoch = np.mean([m['avg_epoch_time'] for m in all_best_metrics])

    print(f"\nAverage Total Training Time : {avg_total_time/60:.1f} phút")
    print(f"Average Time per Epoch     : {avg_epoch:.1f}s")

    f.write(f"\nAverage Total Training Time: {avg_total_time/60:.2f} minutes\n")
    f.write(f"Average Time per Epoch     : {avg_epoch:.2f} seconds\n")
    f.write("\nSeeds used: " + str(seeds))

print(f"\nKết quả đã được lưu vào file:")
print(result_file)


FINAL STATISTICAL RESULTS (5 independent runs)
           E: 0.0335 ± 0.0010
         Sen: 0.9859 ± 0.0031
         Acc: 0.9922 ± 0.0002
        Dice: 0.9830 ± 0.0005

Average Total Training Time : 9.5 phút
Average Time per Epoch     : 28.4s

Kết quả đã được lưu vào file:
/content/drive/MyDrive/DS200_BIG_DATA/result_LUNA/result_swin_dice.txt
